# BF16 Colab validation
A100 80GB required. Read repository README and THIRD_PARTY_NOTICES before running. Qwen Image is research-only. Run cells in order. This notebook is generated from reviewed scripts; original evidence is separate. No Drive or token required.


In [ ]:
import threading
from google.colab import runtime
shutdown_timer=threading.Timer(25*60, runtime.unassign)
shutdown_timer.daemon=True
shutdown_timer.start()
print("25-minute best-effort runtime shutdown armed. Export results promptly.")


In [ ]:
%pip install -q transformers==5.22.1 accelerate==1.15.0 huggingface_hub==1.33.0 safetensors==0.8.0


If Colab asks to restart the session after installation, restart and rerun from the timer cell. Keep Colab supplied PyTorch. Validated environment was Python 3.13.16 / torch 2.14.1+cu130; other environments are not validated.


In [ ]:
# A100 BF16 preflight. No model weights, Drive mount, secrets or public tunnels.
import os, sys, json, time, pathlib, shutil, subprocess, platform, urllib.request
from datetime import datetime, timezone
import torch
import psutil
REPORT_DIR = pathlib.Path("/content/a100_validation")
REPORT_DIR.mkdir(exist_ok=True)
report = {"timestamp_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version.split()[0], "torch": torch.__version__, "gpu": [], "ram_gib": round(psutil.virtual_memory().total / 2**30, 2), "available_ram_gib": round(psutil.virtual_memory().available / 2**30,2), "disk_free_gib": round(shutil.disk_usage("/content").free / 2**30,2)}
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        report["gpu"].append({"name":p.name,"vram_gib":round(p.total_memory/2**30,2),"capability":list(torch.cuda.get_device_capability(i)),"bf16_supported":torch.cuda.is_bf16_supported()})
report["models"] = {}
for repo in ["huihui-ai/Huihui-Qwen3.8-27B-abliterated", "Qwen/Qwen-Image-2.1"]:
    with urllib.request.urlopen("https://huggingface.co/api/models/"+repo, timeout=30) as r:
        data=json.load(r)
    report["models"][repo]={"revision":data.get("sha"),"gated":data.get("gated"),"license":data.get("cardData",{}).get("license"),"safetensors":data.get("safetensors")}
print(json.dumps(report,ensure_ascii=False,indent=2))
(REPORT_DIR/"preflight.json").write_text(json.dumps(report,ensure_ascii=False,indent=2))
assert report["gpu"], "STOP: no GPU. No model download permitted."
assert all("A100" in g["name"] and g["bf16_supported"] for g in report["gpu"]), "STOP: A100 BF16 required."
print("PREFLIGHT_ONLY: no model has been loaded or tested.")


assert all(g["vram_gib"] > 70 for g in report["gpu"]), "STOP: A100 80GB required; 40GB is not validated."


In [ ]:
# Non-commercial technical evaluation under the Qwen Research License.
# Official checkpoint; every learned component remains BF16 (no INT8/FP8/NVFP4).
import subprocess, sys, importlib, time, json, collections, hashlib, gc
if "p" in globals():
    del p
    gc.collect()
    torch.cuda.empty_cache()
print("GPU_ALLOCATED_BEFORE_IMAGE_GIB",round(torch.cuda.memory_allocated()/2**30,3))
assert torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory/2**30 > 70, "STOP: A100 80GB required before model download."
DIFFUSERS_COMMIT="84532a058702c74ae5d1dd6ceb05683fc5d5e768"
subprocess.check_call([sys.executable,"-m","pip","install","-q","git+https://github.com/huggingface/diffusers.git@"+DIFFUSERS_COMMIT])
from diffusers import QwenImage21Pipeline
import diffusers, numpy as np
MODEL_ID="Qwen/Qwen-Image-2.1"
REVISION="d26bb61231c349cf6b7896fa83353113880e1ba3"
assert torch.cuda.is_available() and "A100" in torch.cuda.get_device_name(0)
assert torch.cuda.memory_allocated()/2**30 < 2, "STOP: previous model still occupies GPU."
start=time.monotonic()
torch.cuda.reset_peak_memory_stats()
pipe=QwenImage21Pipeline.from_pretrained(MODEL_ID,revision=REVISION,dtype=torch.bfloat16,token=False).to("cuda")
components={}
for name,module in pipe.components.items():
    if isinstance(module,torch.nn.Module):
        dtype_count=collections.Counter()
        device_count=collections.Counter()
        for p in module.parameters():
            dtype_count[str(p.dtype)]+=p.numel()
            device_count[str(p.device)]+=p.numel()
        components[name]={"parameters_by_dtype":dict(dtype_count),"parameters_by_device":dict(device_count),"quantized":bool(getattr(module,"is_quantized",False))}
        assert set(dtype_count)=={"torch.bfloat16"}, f"STOP: unexpected dtype in {name}"
        assert set(device_count)=={"cuda:0"}, f"STOP: off-device weights in {name}"
        assert not components[name]["quantized"], f"STOP: quantization in {name}"
print("IMAGE_COMPONENTS:",json.dumps(components,indent=2))
image_result={"model":MODEL_ID,"revision":REVISION,"diffusers_version":diffusers.__version__,"diffusers_commit":DIFFUSERS_COMMIT,"gpu":torch.cuda.get_device_name(0),"components":components,"load_seconds":round(time.monotonic()-start,2),"tests":[]}
steps_seen=[]
def validate_latents(pipeline, step, timestep, callback_kwargs):
    if step%5==0 or step==39:
        assert torch.isfinite(callback_kwargs["latents"]).all().item(), "STOP: non-finite latents"
    steps_seen.append(int(step))
    return callback_kwargs
prompt='A photorealistic red ceramic teapot on a wooden table beside a small green plant, soft morning window light, crisp detail, a white card clearly labeled "A100 BF16 TEST".'
start=time.monotonic()
image=pipe(prompt=prompt,width=2048,height=2048,num_inference_steps=40,true_cfg_scale=1.0,generator=torch.Generator("cuda").manual_seed(42),callback_on_step_end=validate_latents,callback_on_step_end_tensor_inputs=["latents"]).images[0]
torch.cuda.synchronize()
path=REPORT_DIR/"qwen_image_2_1_bf16_2048.png"
image.save(path)
pixels=np.asarray(image)
assert image.size==(2048,2048)
assert np.isfinite(pixels).all() and float(pixels.std())>2, "STOP: invalid/constant image"
image_result["tests"].append({"task":"text_to_image","prompt":prompt,"width":image.width,"height":image.height,"mode":image.mode,"requested_steps":40,"actual_steps":len(steps_seen),"seed":42,"seconds":round(time.monotonic()-start,2),"pixel_std":float(pixels.std()),"file":path.name,"sha256":hashlib.sha256(path.read_bytes()).hexdigest()})
image_result["peak_allocated_gib"]=round(torch.cuda.max_memory_allocated()/2**30,3)
image_result["peak_reserved_gib"]=round(torch.cuda.max_memory_reserved()/2**30,3)
image_result["status"]="INFERENCE_COMPLETED_PENDING_VISUAL_AND_INDEPENDENT_REVIEW"
(REPORT_DIR/"qwen_image_2_1_bf16.json").write_text(json.dumps(image_result,ensure_ascii=False,indent=2))
print("QWEN_IMAGE_BF16_RESULT:",json.dumps(image_result,ensure_ascii=False,indent=2))
display(image)
del pipe,module,p
gc.collect()
torch.cuda.empty_cache()
print("IMAGE_WEIGHTS_UNLOADED_GPU_ALLOCATED_GIB",round(torch.cuda.memory_allocated()/2**30,3))



Download the ZIP before shutdown. If download fails, use Colab Files to download JSON/PNG manually. Do not leave paid GPU running while troubleshooting export. A PNG filename/hash in JSON does not prove the file was exported.


In [ ]:
# Export only this test's public-safe evidence. No accounts, secrets or weights.
import importlib.metadata, hashlib, json, shutil, pathlib
from google.colab import files
versions={}
for package in ["torch","transformers","accelerate","huggingface_hub","safetensors","diffusers","numpy","Pillow"]:
    try: versions[package]=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: pass
versions["diffusers_source_commit"]="84532a058702c74ae5d1dd6ceb05683fc5d5e768"
(REPORT_DIR/"runtime_versions.json").write_text(json.dumps(versions,indent=2))
manifest={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in REPORT_DIR.iterdir() if p.is_file() and p.name != "sha256_manifest.json"}
(REPORT_DIR/"sha256_manifest.json").write_text(json.dumps(manifest,indent=2))
archive=shutil.make_archive("/content/a100_validation_evidence","zip",str(REPORT_DIR))
print("EVIDENCE_EXPORT_FILES",sorted(manifest))
print("EXPORT_SHA256",hashlib.sha256(pathlib.Path(archive).read_bytes()).hexdigest())
print("Next: disconnect and delete this paid runtime; then verify zero active sessions.")
files.download(archive)



In [ ]:
from google.colab import runtime
runtime.unassign()
